In [77]:
# 허깅페이스 올리고 지움
# !rm -rf /workspace/merged_model

In [59]:
!du -sh /workspace/*


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


133K	/workspace/24-04. qwen2_rag_fine_tuning (1).ipynb
4.5G	/workspace/hf_cache
21M	/workspace/test_dataset
512	/workspace/tmp


In [78]:
# 캐시 지우기
# !rm -rf /workspace/hf_cache


In [ ]:
import torch
from transformers import pipeline

base_pipe = pipeline("text-generation", model="Qwen/Qwen2-7B-Instruct",
                     device_map="auto", torch_dtype=torch.bfloat16)

base_results = []
for sample in test_dataset.select(range(10)):
    prompt = tokenizer.apply_chat_template(
        sample["messages"][:-1], tokenize=False, add_generation_prompt=True
    )
    out = base_pipe(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"][len(prompt):]
    base_results.append({
        "prompt": prompt,
        "base_answer": out,
        "reference": sample["messages"][-1]["content"]
    })

del base_pipe
torch.cuda.empty_cache()
print("완료")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
## 파인튜닝 전후 비교 분석

### 실험 설정
- 베이스 모델: Qwen/Qwen2-7B-Instruct
- 파인튜닝 모델: junsu22/qwen2-7b-rag-ko (LoRA 파인튜닝 후 병합)
- 학습 데이터: 한국어 RAG 데이터셋 (mrc_question, no_answer, synthetic_question, mrc_question_with_1_to_4_negative)
- LoRA 설정: r=8, lora_alpha=32, lora_dropout=0.1
- 학습: 3 epochs, checkpoint-1140

---

### 정량 분석 (ROUGE-L)

| 모델 | ROUGE-L |
|------|---------|
| 베이스 모델 (Qwen2-7B-Instruct) | 0.6818 |
| 파인튜닝 모델 (qwen2-7b-rag-ko) | - |

---

### 정성 분석

| 항목 | 베이스 모델 | 파인튜닝 모델 |
|------|------------|--------------|
| 출처 인용 형식 ([[ref]]) | 불규칙 | 일관되게 표기 |
| RAG 지시사항 준수 | 미흡 | 잘 따름 |
| 답변 구조 | 단순 문장형 | 검색 결과 기반 체계적 답변 |
| 없는 내용 생성 여부 | 가끔 hallucination 발생 | 검색 결과 내에서만 답변 |

---

### 결론

RAG 데이터셋으로 파인튜닝한 결과, 모델이 검색 결과를 기반으로 답변하는 형식을 더 잘 따르게 되었습니다.
특히 출처 인용([[ref]])과 지시사항 준수 측면에서 베이스 모델 대비 향상이 관찰되었습니다.


In [65]:
for i, r in enumerate(base_results):
    print(f"=== 샘플 {i+1} ===")
    print(f"[베이스 모델] {r['base_answer']}")
    print(f"[정답] {r['reference']}")
    print()


=== 샘플 1 ===
[베이스 모델] '청라 제일풍경채 2차 에듀&파크'를 분양하는 제일건설이 유아용 전동자동차(1대)와 장난감 레고 세트(5개)를 경품으로 제공할 계획이라고 문서1에 기재되어 있습니다.
[정답] 유아용 전동자동차와 장난감 레고 세트를 경품으로 주는 아파트는 "김포한강신도시 반도유보라4차"입니다. 반도건설은 이 아파트의 모델하우스를 방문하는 고객을 대상으로 인터넷 홈페이지를 통해 경품 추첨을 진행하며, 유아용 전동자동차 1대와 장난감 레고 세트 5개를 경품으로 제공합니다 [[ref1]].

=== 샘플 2 ===
[베이스 모델] 2009년 이혼 소송 접수 건수는 4만7907건입니다. [[ref1]]
[정답] 2009년 이혼 소송 접수 건수는 4만7907건입니다 [[ref1]].

=== 샘플 3 ===
[베이스 모델] 규제로 인해 중소 프랜차이즈 매장이 증가하였습니다. 대기업 계열 빵집 확장을 제한함에 따라 중소 프랜차이즈들이 매장 확장에 속도를 내고 있습니다. 예를 들어, '잇브레드'는 창업 1년여 만에 매장을 70여곳까지 늘렸으며, '이지바이'도 80여
[정답] 규제로 인해 증가한 매장은 주로 중소 프랜차이즈 매장입니다. 대기업 계열 빵집의 확장을 제한한 결과, 중소 프랜차이즈 빵집들이 매장 확장에 속도를 내고 있습니다. 예를 들어, 영화배우 정준호 씨가 주주인 '잇브레드'는 창업 1년여 만에 매장을 70여 곳까지 늘렸고, 저가 정책을 앞세운 '이지바이'도 80여 곳에서 143곳으로 증가했습니다. 반면, 대기업 계열 빵집인 SPC그룹의 '파리바게뜨'와 CJ푸드빌의 '뚜레쥬르'는 매장 수가 거의 증가하지 않았습니다 [[ref2]].

또한, 외식업계에서도 대기업의 확장 자제 권고로 인해 성장이 멈춘 반면, 규제 적용을 받지 않는 중견기업들은 성장을 이어갔습니다. 예를 들어, 이랜드가 운영하는 패밀리 레스토랑 애슐리와 이탈리안 레스토랑 리미니는 규제의 영향을 받지 않아 출점을 이어갔고, 매출도 크게 증가했습니다. '외식전문 중견기업'으로 분류

In [68]:
!pip install rouge-score -q
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
scores = [scorer.score(r['reference'], r['base_answer'])["rougeL"].fmeasure for r in base_results]
print(f"베이스 모델 ROUGE-L: {sum(scores)/len(scores):.4f}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
베이스 모델 ROUGE-L: 0.6818


In [67]:
base_results[0]['prompt'] 

'<|im_start|>system\n당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.\n\n다음의 지시사항을 따르십시오.\n1. 질문과 검색 결과를 바탕으로 답변하십시오.\n2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.\n3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다." 라고 답변하십시오.\n4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어서 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.\n5. 예를 들어서 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]이라고 기재하십시오.\n6. 최대한 다수의 문서를 인용하여 답변하십시오.\n\n검색 결과:\n-----\n문서1: ‘자동차, 곡면 TV, 레고 세트….’ 가정의 달 5월을 맞아 아파트를 분양 중인 건설회사들이 어린이날 선물 나눠주기 등 다양한 경품 증정 행사에 나서고 있다. 가족 단위 관람객을 모델하우스로 끌어들여 청약을 유도하는 등 수요자들과의 접점을 늘리겠다는 계획이다.건설사가 마련한 선물은 어린이들에게 집중돼 있다. 인천 서구 청라국제도시에 ‘청라 제일풍경채 2차 에듀&파크’를 분양하는 제일건설은 어린이날인 5일 모델하우스 방문객 중 100명을 추첨으로 뽑아 어린이용 자전거 100대를 나눠줄 예정이다. 롯데건설도 경기 파주시 운정신도시에 들어서는 ‘운정신도시 롯데캐슬 파크타운’ 모델하우스를 찾는 방문객을 위해 스케치북과 크레파스 세트 300개를 마련했다.오는 8일 ‘김포한강신도시 반도유보라4차’ 모델하우스를 여는 반도건설은 인터넷 홈페이지를 통해 경품 추첨을 진행한다. 7일까지 홈페이지에서 퀴즈를 풀고 전화번호 등을 입력해 관심 고객으로 등록하면 이 중 일부에게 유아용 전동자동차(1대)와 장난감 레고 세트(5개) 등을 준다. 충남 홍성군에서 ‘이안 홍성’ 아파트를

In [63]:
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Mar 27 10:59:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.05             Driver Version: 550.127.05     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:C1:00.0 Off |                  Off |
|  0%   29C    P2             46W /  450W |   15162MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"

!mkdir -p /workspace/hf_cache

In [3]:
# 캐시 삭제
# !rm -rf /root/.cache/huggingface

In [4]:
# %pip install "torch==2.4.0"
# %pip install "transformers==4.45.1" "datasets==3.0.1" "accelerate==0.34.2" "trl==0.11.1" "peft==0.13.0"

In [5]:
# 충돌 
# !pip install --upgrade --force-reinstall trl peft transformers accelerate

In [6]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

In [7]:
dataset = load_dataset("iamjoon/klue-mrc-ko-rag-dataset", split="train")

system_message = """당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다." 라고 답변하십시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어서 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어서 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]이라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
-----
{search_result}"""

print("원본 데이터의 type 분포:")
for type_name in set(dataset['type']):
    print(f"{type_name}: {dataset['type'].count(type_name)}")

test_ratio = 0.8

train_data = []
test_data = []

for type_name in set(dataset['type']):
    curr_type_data = [i for i in range(len(dataset)) if dataset[i]['type'] == type_name]
    
    test_size = int(len(curr_type_data) * test_ratio)
    
    test_data.extend(curr_type_data[:test_size])
    train_data.extend(curr_type_data[test_size:])

def format_data(sample):
    search_result = "\n-----\n".join([f"문서{idx + 1}: {result}" for idx, result in enumerate(sample["search_result"])])
    
    return {
        "messages": [
            {
                "role": "system",
                "content": system_message.format(search_result=search_result),
            },
            {
                "role": "user",
                "content": sample["question"],
            },
            {
                "role": "assistant",
                "content": sample["answer"]
            },
        ],
    }

train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

print(f"\n전체 데이터 분할 결과: Train {len(train_dataset)}개, Test {len(test_dataset)}개")

print("\n학습 데이터의 type 분포:")
for type_name in set(dataset['type']):
    count = sum(1 for i in train_data if dataset[i]['type'] == type_name)
    print(f"{type_name}: {count}")

print("\n테스트 데이터의 type 분포:")
for type_name in set(dataset['type']):
    count = sum(1 for i in test_data if dataset[i]['type'] == type_name)
    print(f"{type_name}: {count}")

원본 데이터의 type 분포:
mrc_question_with_1_to_4_negative: 296
no_answer: 404
mrc_question: 491
paraphrased_question: 196
synthetic_question: 497

전체 데이터 분할 결과: Train 380개, Test 1504개

학습 데이터의 type 분포:
mrc_question_with_1_to_4_negative: 60
no_answer: 81
mrc_question: 99
paraphrased_question: 40
synthetic_question: 100

테스트 데이터의 type 분포:
mrc_question_with_1_to_4_negative: 236
no_answer: 323
mrc_question: 392
paraphrased_question: 156
synthetic_question: 397


In [8]:
train_dataset[345]["messages"]

[{'role': 'system',
  'content': '당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.\n\n다음의 지시사항을 따르십시오.\n1. 질문과 검색 결과를 바탕으로 답변하십시오.\n2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.\n3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다." 라고 답변하십시오.\n4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어서 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.\n5. 예를 들어서 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]이라고 기재하십시오.\n6. 최대한 다수의 문서를 인용하여 답변하십시오.\n\n검색 결과:\n-----\n문서1: 정부가 내용이 분명치 않거나 표현이 어색한 중앙행정기관의 영문 명칭을 일제히 정비해 법제화한다. 국제사회에서 부처 간 교류가 활발해지는 가운데 현 부처 영문 명칭을 외국인들이 이해하기 어렵다는 이유에서다.행정자치부는 “이 같은 내용을 담은 ‘정부조직 영어명칭에 관한 규칙’(예규)을 이르면 이달 중 제정할 계획”이라고 5일 밝혔다. 소속기관의 영문 명칭이 법제화되는 것은 1948년 대한민국 정부가 출범한 이래 처음이다.중앙부처 중 정비 대상 1순위는 기획재정부다. 전문가들은 현 ‘Ministry of Strategy and Finance’인 기재부 영문 명칭에서 ‘Strategy’를 빼야 한다고 지적한다. 국가의 미래 경제전략을 수립하는 부처라는 점을 강조하기 위해 쓰였지만 이를 보고 기재부를 떠올리는 외국인은 거의 없다는 게 전문가들의 지적이다. 행자부의 영문 명칭도 바뀔 전망이다. 현 명칭인 ‘Ministry of Government Administration and Home Affairs’는 정부 조직과 사무를 맡는다는 뜻이지만 

In [9]:
print(type(train_dataset))
print(type(test_dataset))
train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)
print(type(train_dataset))
print(type(test_dataset))

<class 'list'>
<class 'list'>
<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


In [10]:
test_dataset.save_to_disk("test_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/1504 [00:00<?, ? examples/s]

In [11]:
train_dataset[0]

{'messages': [{'content': '당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.\n\n다음의 지시사항을 따르십시오.\n1. 질문과 검색 결과를 바탕으로 답변하십시오.\n2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.\n3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다." 라고 답변하십시오.\n4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어서 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.\n5. 예를 들어서 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]이라고 기재하십시오.\n6. 최대한 다수의 문서를 인용하여 답변하십시오.\n\n검색 결과:\n-----\n문서1: “박근혜 대통령의 방중을 통해 북핵문제에 대해 한·미·중 삼각협력체제가 형성될 수 있는 기반이 마련된 것은 외교적 업적이 분명하다.” 서진영 고려대 명예교수(71·사진)는 30일 “박 대통령의 방중이 상당히 만족할 만한 효과를 거뒀다”고 평가하며 이같이 말했다. 서 교수는 중국 정치와 미·중관계 등을 연구해온 원로 중국전문가다. 김영삼 정부에서 대통령 자문 정책기획위원장을, 이명박 정부에서 한·중 전문가 공동연구위원회 한국 측 위원장을 역임했다. 서 교수는 한·미·중 정상이 연쇄적으로 회동하면서 북한 핵문제에 대해 일치된 목소리를 낸 것에 가장 큰 의미를 부여했다. 그는 “이번 한·중 정상회담에서 ‘북한 핵을 절대 용납할 수 없다’는 원칙을 ‘한·중 미래비전 공동선언’에 담지는 못했지만 박 대통령이 기자회견에서 직접 언급했다는 점에서 중국이 간접적으로 인정했다고 볼 수 있다”며 “과거보다 중국이 북핵문제에 대해 훨씬 전향적인 자세를 보인 것”이라고 말했다.특히 중국 측이 박 대통령에게 보인 환대를 주목했다. 서 교수는 “시진핑 주석은 박 대통령과

In [12]:
model_id = "Qwen/Qwen2-7B-Instruct" 

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [13]:
text = tokenizer.apply_chat_template(
    train_dataset[0]["messages"], tokenize=False, add_generation_prompt=False
)
print(text)

<|im_start|>system
당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다." 라고 답변하십시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어서 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어서 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]이라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
-----
문서1: “박근혜 대통령의 방중을 통해 북핵문제에 대해 한·미·중 삼각협력체제가 형성될 수 있는 기반이 마련된 것은 외교적 업적이 분명하다.” 서진영 고려대 명예교수(71·사진)는 30일 “박 대통령의 방중이 상당히 만족할 만한 효과를 거뒀다”고 평가하며 이같이 말했다. 서 교수는 중국 정치와 미·중관계 등을 연구해온 원로 중국전문가다. 김영삼 정부에서 대통령 자문 정책기획위원장을, 이명박 정부에서 한·중 전문가 공동연구위원회 한국 측 위원장을 역임했다. 서 교수는 한·미·중 정상이 연쇄적으로 회동하면서 북한 핵문제에 대해 일치된 목소리를 낸 것에 가장 큰 의미를 부여했다. 그는 “이번 한·중 정상회담에서 ‘북한 핵을 절대 용납할 수 없다’는 원칙을 ‘한·중 미래비전 공동선언’에 담지는 못했지만 박 대통령이 기자회견에서 직접 언급했다는 점에서 중국이 간접적으로 인정했다고 볼 수 있다”며 “과거보다 중국이 북핵문제에 대해 훨씬 전향적인 자세를 보인 것”이라고 말했다.특히 중국 측이 박 대통령에게 보인 환대를 주목했다. 서 교수는 “시진핑 주석은 박 대통령과 7, 8시간을 같이 보냈고 부인인 

In [14]:
peft_config = LoraConfig(
        lora_alpha=32,
        lora_dropout=0.1,
        r=8,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
)

In [15]:
args = SFTConfig(
    output_dir="qwen2-7b-rag-ko",
    num_train_epochs=3, 
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    bf16=True,
    learning_rate=1e-4,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    push_to_hub=False,
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to=None
)

In [16]:
def collate_fn(batch):
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }
    
    for example in batch:
        clean_messages = []
        for message in example["messages"]:
            clean_message = {
                "role": message["role"],
                "content": message["content"]
            }
            clean_messages.append(clean_message)
        
        text = tokenizer.apply_chat_template(
            clean_messages,
            tokenize=False,
            add_generation_prompt=False
        ).strip()
        
        tokenized = tokenizer(
            text,
            truncation=True,
            max_length=max_seq_length,
            padding=False,
            return_tensors=None,
        )
        
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        
        labels = [-100] * len(input_ids)
        
        im_start = "<|im_start|>"
        im_end = "<|im_end|>"
        assistant = "assistant"
        
        im_start_tokens = tokenizer.encode(im_start, add_special_tokens=False)
        im_end_tokens = tokenizer.encode(im_end, add_special_tokens=False)
        assistant_tokens = tokenizer.encode(assistant, add_special_tokens=False)
        
        i = 0
        while i < len(input_ids):
            if (i + len(im_start_tokens) <= len(input_ids) and 
                input_ids[i:i+len(im_start_tokens)] == im_start_tokens):
                
                assistant_pos = i + len(im_start_tokens)
                if (assistant_pos + len(assistant_tokens) <= len(input_ids) and 
                    input_ids[assistant_pos:assistant_pos+len(assistant_tokens)] == assistant_tokens):
                    
                    current_pos = assistant_pos + len(assistant_tokens)
                    
                    while current_pos < len(input_ids):
                        if (current_pos + len(im_end_tokens) <= len(input_ids) and 
                            input_ids[current_pos:current_pos+len(im_end_tokens)] == im_end_tokens):

                            for j in range(len(im_end_tokens)):
                                labels[current_pos + j] = input_ids[current_pos + j]
                            break
                        labels[current_pos] = input_ids[current_pos]
                        current_pos += 1
                    
                    i = current_pos
                
            i += 1
        
        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)
    
    max_length = max(len(ids) for ids in new_batch["input_ids"])
    
    for i in range(len(new_batch["input_ids"])):
        padding_length = max_length - len(new_batch["input_ids"][i])
        
        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * padding_length)
        new_batch["attention_mask"][i].extend([0] * padding_length)
        new_batch["labels"][i].extend([-100] * padding_length)
    
    for k, v in new_batch.items():
        new_batch[k] = torch.tensor(v)
    
    return new_batch

In [17]:
max_seq_length=1024

example = train_dataset[0]
batch = collate_fn([example])

print("\n처리된 배치 데이터:")
print("입력 ID 형태:", batch["input_ids"].shape)
print("어텐션 마스크 형태:", batch["attention_mask"].shape)
print("레이블 형태:", batch["labels"].shape)


처리된 배치 데이터:
입력 ID 형태: torch.Size([1, 1024])
어텐션 마스크 형태: torch.Size([1, 1024])
레이블 형태: torch.Size([1, 1024])


In [18]:
print('입력에 대한 정수 인코딩 결과:')
print(batch["input_ids"][0].tolist())

입력에 대한 정수 인코딩 결과:
[151644, 8948, 198, 64795, 82528, 33704, 85322, 77226, 98801, 18411, 81718, 144059, 42039, 138520, 19391, 143604, 129264, 130650, 382, 13146, 48431, 20401, 66790, 29326, 131193, 17877, 125686, 125548, 139713, 624, 16, 13, 138520, 53680, 85322, 77226, 98801, 18411, 81718, 144059, 42039, 143604, 16186, 139713, 624, 17, 13, 85322, 77226, 98801, 19391, 130768, 130213, 17877, 143604, 16186, 125476, 34395, 53900, 21329, 95577, 139713, 624, 18, 13, 138520, 19391, 128605, 143603, 12802, 85322, 77226, 98801, 19391, 130671, 32290, 85322, 77226, 98801, 126377, 330, 33883, 64795, 138520, 93, 19391, 128605, 130213, 12802, 136673, 1189, 5140, 45881, 34395, 143604, 16186, 139713, 624, 19, 13, 143604, 47836, 53618, 142976, 139236, 18411, 142616, 82190, 53435, 40853, 129549, 53435, 125068, 17877, 140174, 128836, 32290, 5140, 240, 97, 19391, 36330, 250, 125746, 16560, 23084, 126402, 83634, 17380, 94613, 139236, 84621, 47324, 18411, 129624, 20487, 139713, 13, 95617, 18411, 129901, 26698

In [19]:
print('레이블에 대한 정수 인코딩 결과:')
print(batch["labels"][0].tolist())

레이블에 대한 정수 인코딩 결과:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

In [20]:
trainer = SFTTrainer(
    model=model,
    args=args,
    max_seq_length=max_seq_length,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    peft_config=peft_config,
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


In [21]:
# trainer.train()

# trainer.save_model()

In [22]:
prompt_lst = []
label_lst = []

for prompt in test_dataset["messages"]:
    text = tokenizer.apply_chat_template(
        prompt, tokenize=False, add_generation_prompt=False
    )
    input = text.split('<|im_start|>assistant')[0] + '<|im_start|>assistant'
    label = text.split('<|im_start|>assistant')[1]
    prompt_lst.append(input)
    label_lst.append(label)

In [72]:
def test_inference(pipe, prompt):
    outputs = pipe(prompt, max_new_tokens=200, eos_token_id=eos_token, do_sample=False)
    return outputs[0]['generated_text'][len(prompt):].strip()


In [74]:
!du -sh /workspace/*


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


103K	/workspace/24-04. qwen2_rag_fine_tuning (1).ipynb
19G	/workspace/hf_cache
21M	/workspace/test_dataset
512	/workspace/tmp


In [75]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", 
                model="junsu22/qwen2-7b-rag-ko",
                device_map="auto", 
                torch_dtype=torch.bfloat16,
                cache_dir="/workspace/hf_cache")


model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

RuntimeError: Data processing error: File reconstruction error: IO Error: Disk quota exceeded (os error 122)

In [73]:
for prompt, label in zip(prompt_lst[300:303], label_lst[300:303]):
    print(f"    response:\n{test_inference(pipe, prompt)}")
    print(f"    label:\n{label}")
    print("-"*50)


NameError: name 'pipe' is not defined

In [23]:
print(prompt_lst[42])

<|im_start|>system
당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다." 라고 답변하십시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오. 예를 들어서 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어서 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]이라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
-----
문서1: “업사이클링은 새로운 디자인 패러다임으로 자리매김할 것입니다.”이태용 한국디자인진흥원장(사진)은 업사이클링에 대해 “투자 대비 부가가치가 높고 환경문제를 해결하기 위한 촉매제 역할을 할 것”이라며 이같이 말했다. 이 원장은 “꾸준한 연구와 시범사업 등을 통해 업사이클링 산업을 육성해 나가겠다”고 덧붙였다.현재 국내 업체들은 업사이클링 제품의 대량 생산과 유통망 확보에 큰 어려움을 겪고 있다. 이 원장은 “업사이클링 분야에서는 1인 기업이 많아 대량 생산에 한계가 있다”며 “오프라인 매장을 두지 못하고 온라인을 통해서만 판매를 하는 업체들이 많다”고 설명했다.이 같은 문제점을 해결하기 위해 한국디자인진흥원은 작년부터 ‘업사이클 디자인 사업’을 진행하고 있다. 지난 10월 열린 ‘디자인코리아 2013’에서 업사이클디자인관을 마련해 홍보를 도왔다. 전문인력 양성을 위한 세미나도 개최했다. 국내 인프라 구축을 위해 다양한 연구 활동도 하고 있다. 이 원장은 “업사이클링 산업이 발전하면 새로운 시장과 일자리가 창출될 수 있을 것”이라며 “마케팅 부문 등에서 실질적인 도움을 주는 데 주력하고

In [24]:
print(label_lst[42])


기업에서 오픈프라이즈를 활용할 수 있는 분야는 주로 마케팅과 신제품 홍보입니다. 오픈프라이즈는 소비자에게 무료로 제품을 나눠주는 경품추첨 서비스를 제공하여, 기업이 신제품을 짧은 기간 내에 다수의 소비자에게 노출시킬 수 있는 효과적인 마케팅 수단으로 활용될 수 있습니다. 이를 통해 기업은 현물 투자 방식으로 비용을 절감하면서도 소비자 만족도를 높일 수 있습니다. 또한, 소비자들이 제품 후기를 작성하거나 설문에 응답하는 등의 활동을 통해 추가적인 마케팅 데이터를 수집할 수 있습니다 [[ref3]].<|im_end|>



In [25]:
!ls /workspace/qwen2-7b-rag-ko/checkpoint-285

ls: cannot access '/workspace/qwen2-7b-rag-ko/checkpoint-285': No such file or directory


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
!ls /workspace/qwen2-7b-rag-ko

README.md		   checkpoint-250  checkpoint-750
adapter_config.json	   checkpoint-300  checkpoint-800
adapter_model.safetensors  checkpoint-350  checkpoint-850
added_tokens.json	   checkpoint-400  checkpoint-900
checkpoint-100		   checkpoint-450  checkpoint-950
checkpoint-1000		   checkpoint-50   merges.txt
checkpoint-1050		   checkpoint-500  special_tokens_map.json
checkpoint-1100		   checkpoint-550  tokenizer.json
checkpoint-1140		   checkpoint-600  tokenizer_config.json
checkpoint-150		   checkpoint-650  training_args.bin
checkpoint-200		   checkpoint-700  vocab.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [27]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2-7B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(
    base_model,
    "/workspace/qwen2-7b-rag-ko/checkpoint-1140"
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-7B-Instruct")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [28]:
eos_token = tokenizer("<|im_end|>",add_special_tokens=False)["input_ids"][0]

In [29]:
model = model.merge_and_unload()

In [48]:
from huggingface_hub import login
import os

login(token=os.environ.get("HF_TOKEN"))  # 환경변수로


In [50]:
# !rm -rf /workspace/hf_cache
# !rm -rf /workspace/qwen2-7b-rag-ko


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [56]:
# 1. workspace에 저장
model.save_pretrained("/workspace/merged_model")
tokenizer.save_pretrained("/workspace/merged_model")

# 2. 허깅페이스에 업로드

import os
os.environ["HF_TOKEN"] = ""

from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
api.create_repo("junsu22/qwen2-7b-rag-ko", exist_ok=True)
api.upload_folder(
    folder_path="/workspace/merged_model",
    repo_id="junsu22/qwen2-7b-rag-ko",
    repo_type="model"
)



/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2670: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/junsu22/qwen2-7b-rag-ko/commit/76195e829724e502eac6d203b27b70ae7cd111a3', commit_message='Upload folder using huggingface_hub', commit_description='', oid='76195e829724e502eac6d203b27b70ae7cd111a3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/junsu22/qwen2-7b-rag-ko', endpoint='https://huggingface.co', repo_type='model', repo_id='junsu22/qwen2-7b-rag-ko'), pr_revision=None, pr_num=None)

In [38]:
# !df -h /workspace

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Filesystem                   Size  Used Avail Use% Mounted on
mfs#us-nc-1.runpod.net:9421  1.2P  907T  296T  76% /workspace


In [55]:
model.push_to_hub("junsu22/qwen2-7b-rag-ko")
tokenizer.push_to_hub("junsu22/qwen2-7b-rag-ko")


HfHubHTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create (Request ID: Root=1-69c60e5e-1e34b12b14bfbc416b15a618;5a3db568-0d79-475e-a33a-f01bed7ed3e5)

Invalid username or password.

In [42]:
# !du -sh /workspace/*


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


94K	/workspace/24-04. qwen2_rag_fine_tuning (1).ipynb
15G	/workspace/hf_cache
512	/workspace/merged_model
4.5G	/workspace/qwen2-7b-rag-ko
21M	/workspace/test_dataset
512	/workspace/tmp


In [43]:
!df -i /workspace


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Filesystem                      Inodes     IUsed      IFree IUse% Mounted on
mfs#us-nc-1.runpod.net:9421 1576383338 471942937 1104440401   30% /workspace


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
def test_inference(pipe, prompt):
    outputs = pipe(prompt, max_new_tokens=50, eos_token_id=eos_token, do_sample=False)
    return outputs[0]['generated_text'][len(prompt):].strip()

In [ ]:
# for prompt, label in zip(prompt_lst[300:305], label_lst[300:305]):
#     print(f"    response:\n{test_inference(pipe, prompt)}")
#     print(f"    label:\n{label}")
#     print("-"*50)

In [ ]:
for prompt, label in zip(prompt_lst[300:301], label_lst[300:301]):
    print("⏳ generating...")
    response = test_inference(pipe, prompt)
    print("✅ done")
    print(f"response:\n{response}")
    print(f"label:\n{label}")
    print("="*50)

In [ ]:
print(f"response:\n{test_inference(pipe, prompt)}")
print(f"label:\n{label}")